# Preparação do ambiente

In [0]:
## atualizado em 2026-02-13 (ntb algoritmo)

In [0]:
!pip install -q \
    "pydantic<2" spacy==3.7.4 \
    sentence-transformers==3.0.1 \
    numpy==1.26.4 \
    striprtf==0.0.22 \
    tqdm==4.66.5 \
    html2text \
    ftfy 

In [0]:
!pip install lxml

In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("environment", "dev", "Environment")
environment = dbutils.widgets.get("environment")

In [0]:
if environment in ["hml", "prd"]:
    environment_tbl = ""
else:
    environment_tbl = f"{environment}_"

In [0]:
OUTPUT_TABLENAME = f"{environment_tbl}tb_diamond_mod_colon_entrada" #Tabela legada ia.tb_diamond_mod_colon alterado dia 26/03/2026
VW_BALIZADORA = f"{environment_tbl}vw_colon_balizador"
SCHEMA = "ia"

In [0]:
print('Environment:   ', environment_tbl)
print('Tabela destino:', OUTPUT_TABLENAME)
print('VW balizadora: ', VW_BALIZADORA)
print('VW balizadora: ', SCHEMA)

# Criação das tabelas e estruturas

In [0]:
# spark.sql(f"""
# CREATE OR REPLACE TEMP VIEW {VW_BALIZADORA} AS
# WITH base AS (
#   SELECT
#       e.id_unidade                                     AS idunidade,
#       e.nme_hospital                                   AS unidade,
#       p.id_paciente                                    AS id_pct,
#       p.cli_nome                                       AS paciente,
#       p.cli_genero                                     AS sexo,
#       p.cli_data_nascimento                            AS nascimento,
#       p.doc_cpf                                        AS cpf,
#       p.cli_telefone_numero                            AS telefonePaciente,
#       p.cli_telefone_ddd                               AS telefonePacienteDDD,
#       e.tp_procedimento                                AS procedimento,
#       e.num_pedido_integracao                          AS num_pedido_integracao,
#       e.nme_regional_hospital                          AS nme_regional_hospital,
#       e.nme_convenio                                   AS nme_convenio,

#       regexp_replace(lower(coalesce(e.dsc_procedimento_integracao, cast(e.cod_procedimento as string), '')), '[\\u00A0\\u2007\\u202F]', ' ') as exame_nr,

#       /*calculo o numero de meses entre as duas datas e divido por 12 para ter o valor em anos*/
#       FLOOR(months_between(DATE(e.dt_dia_exame), DATE(p.cli_data_nascimento)) / 12) AS idade_anos,
#       CASE
#         WHEN exame_nr RLIKE '(tomografia|angio|tc|tomo)'
#           THEN 'CT'
#         ELSE 'IND'
#       END                                              AS modalidade,
#       translate(
#         regexp_replace(
#           lower(coalesce(exame_nr, cast(e.cod_procedimento as string), '')),
#           '[\\u00A0\\u2007\\u202F]', ' '
#         ),
#         'áàãâäéèêëíìîïóòõôöúùûüç',
#         'aaaaaeeeeiiiiooooouuuuc'
#       ) AS exame,

#       e.dt_dia_exame                                   AS dataexame,
#       translate(
#         regexp_replace(
#           lower(coalesce(e.nme_origem_exame, e.tp_proced_classe_encontro, '')),
#           '[\\u00A0\\u2007\\u202F]', ' '
#         ),
#         'áàãâäéèêëíìîïóòõôöúùûüç',
#         'aaaaaeeeeiiiiooooouuuuc'
#       ) AS tipoexame,

#       CAST(e.id_exame AS STRING)                       AS an,

#       e.dt_liberacao_laudo                             AS DataLaudoLiberado,
#       array_join(
#         transform(e.proced_lista_exames, x -> x.laudo_transformado),
#         '\n'
#       ) AS Laudo,
#       COALESCE(e.dt_liberacao_laudo, e.dt_previsao_laudo ) AS DataLaudoFinal,
#       CASE
#         WHEN e.dt_liberacao_laudo IS NOT NULL THEN 5
#         WHEN e.dt_previsao_laudo  IS NOT NULL THEN 4
#         ELSE 0
#       END AS idstatus

#   FROM gold_corporativo_ia.corporativo.tb_gold_mov_exame   e
#   LEFT JOIN gold_corporativo_ia.corporativo.tb_gold_mov_paciente p
#          ON p.id_paciente = e.id_patient
# ),
# filtrada AS (
#   SELECT *
#   FROM base
#   WHERE
#     UPPER(trim(procedimento)) IN  ('IMG', 'IMA')
#     AND LOWER(exame_nr) ILIKE '%tomografia%'
#     OR LOWER(exame_nr) ILIKE '%tc'
#     AND(
#       LOWER(exame_nr) ILIKE '%abd%' OR LOWER(exame_nr) ILIKE '%pelve%'
#     )
#     AND LOWER(exame_nr) not ilike '%angio%'
#     AND LOWER(exame_nr) not ilike '%superior%'   
#     AND DataLaudoFinal IS NOT NULL
#     AND DATE(DataLaudoFinal) BETWEEN date_sub(current_date(), 30)
#                                  AND date_sub(current_date(), 1) -- ontem
# ),

# dedup AS (
#   SELECT
#     f.*,
#     ROW_NUMBER() OVER (
#       PARTITION BY an
#       ORDER BY TIMESTAMP(DataLaudoFinal) DESC
#     ) AS rn,
#     CASE WHEN ROW_NUMBER() OVER (
#              PARTITION BY an
#              ORDER BY TIMESTAMP(DataLaudoFinal) DESC
#          ) = 1
#          THEN 1 ELSE 0 END AS an_duplicado
#   FROM filtrada f
# )
# SELECT
#   idunidade,
#   unidade,
#   id_pct,
#   paciente,
#   telefonePacienteDDD,
#   telefonePaciente,
#   sexo,
#   nascimento,
#   idade_anos,
#   cpf,
#   modalidade,
#   exame,
#   num_pedido_integracao,
#   nme_regional_hospital,
#   nme_convenio,
#   dataexame,
#   tipoexame,
#   an,
#   an_duplicado,
#   idstatus,
#   -- 4 AS idstatus,                 -- mantido
#   -- idLaudo,                       
#   DataLaudoFinal AS DataLaudoLiberado,  -- >>> usa a data escolhida
#   Laudo,
#   -- current_timestamp() as datCarga
#   CAST(from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo') AS TIMESTAMP_NTZ) AS dataExecucaoModelo
# FROM dedup
# """)

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW {VW_BALIZADORA} AS
WITH base AS (
  SELECT
      e.id_unidade                                     AS idunidade,
      e.nme_hospital                                   AS unidade,
      p.id_paciente                                    AS id_pct,
      p.cli_nome                                       AS paciente,
      p.cli_genero                                     AS sexo,
      p.cli_data_nascimento                            AS nascimento,
      p.doc_cpf                                        AS cpf,
      p.cli_telefone_numero                            AS telefonePaciente,
      p.cli_telefone_ddd                               AS telefonePacienteDDD,
      e.tp_procedimento                                AS procedimento,
      e.num_pedido_integracao                          AS num_pedido_integracao,
      e.nme_regional_hospital                          AS nme_regional_hospital,
      e.nme_convenio                                   AS nme_convenio,

      regexp_replace(lower(coalesce(e.proced_descricao_ajustado, cast(e.cod_procedimento as string), '')), '[\\u00A0\\u2007\\u202F]', ' ') as exame_nr,

      regexp_replace(lower(coalesce(e.dsc_codigo, '')), '[\\u00A0\\u2007\\u202F]', ' ') AS dsc_codigo_txt,
      regexp_replace(lower(coalesce(cast(e.cod_procedimento as string), '')), '[\\u00A0\\u2007\\u202F]', ' ') AS cod_procedimento_txt,

      -- calculo o numero de meses entre as duas datas e divido por 12 para ter o valor em anos/
      FLOOR(months_between(DATE(e.dt_dia_exame), DATE(p.cli_data_nascimento)) / 12) AS idade_anos,
      CASE
        WHEN exame_nr RLIKE '(tomografia|angio|tc|tomo)'
          THEN 'CT'
        WHEN exame_nr RLIKE '(colonoscopia|colono)' AND exame_nr NOT RLIKE '(tc|tomografia|ressonancia|rx)'
          THEN 'COL'
        ELSE 'IND'
      END                                              AS modalidade,
      translate(
        regexp_replace(
          lower(coalesce(exame_nr, cast(e.cod_procedimento as string), '')),
          '[\\u00A0\\u2007\\u202F]', ' '
        ),
        'áàãâäéèêëíìîïóòõôöúùûüç',
        'aaaaaeeeeiiiiooooouuuuc'
      ) AS exame,

      e.dt_dia_exame                                   AS dataexame,
      translate(
        regexp_replace(
          lower(coalesce(e.nme_origem_exame, e.tp_proced_classe_encontro, '')),
          '[\\u00A0\\u2007\\u202F]', ' '
        ),
        'áàãâäéèêëíìîïóòõôöúùûüç',
        'aaaaaeeeeiiiiooooouuuuc'
      ) AS tipoexame,

      CAST(e.id_exame AS STRING)                       AS an,

      e.dt_liberacao_laudo                             AS DataLaudoLiberado,
      array_join(
        transform(e.proced_lista_exames, x -> x.laudo_transformado),
        '\n'
      ) AS Laudo,
      COALESCE(e.dt_liberacao_laudo, e.dt_previsao_laudo ) AS DataLaudoFinal,
      CASE
        WHEN e.dt_liberacao_laudo IS NOT NULL THEN 5
        WHEN e.dt_previsao_laudo  IS NOT NULL THEN 4
        ELSE 0
      END AS idstatus

  FROM gold_corporativo_ia.corporativo.tb_gold_mov_exame   e
  LEFT JOIN gold_corporativo_ia.corporativo.tb_gold_mov_paciente p
         ON p.id_paciente = e.id_patient
),
filtrada AS (
  SELECT *
  FROM base
  WHERE
    UPPER(trim(procedimento)) IN  ('IMG', 'IMA')
    AND(
    (
    LOWER(COALESCE(exame_nr, '')) ILIKE '%tomografia%'
    OR LOWER(COALESCE(exame_nr, '')) ILIKE '%tomo%'
    OR LOWER(COALESCE(exame_nr, '')) ILIKE '%tc%'
    OR LOWER(COALESCE(dsc_codigo_txt, '')) ILIKE '%tomografia%'
    OR LOWER(COALESCE(dsc_codigo_txt, '')) ILIKE '%tomo%'
    OR LOWER(COALESCE(dsc_codigo_txt, '')) ILIKE '%tc%'
    OR LOWER(COALESCE(cod_procedimento_txt, '')) ILIKE '%tomografia%'
    OR LOWER(COALESCE(cod_procedimento_txt, '')) ILIKE '%tomo%'
    OR LOWER(COALESCE(cod_procedimento_txt, '')) ILIKE '%tc%'
    )
    AND(
    LOWER(COALESCE(exame_nr, '')) ILIKE '%abd%'
    OR LOWER(COALESCE(exame_nr, '')) ILIKE '%pelve%'
    OR LOWER(COALESCE(dsc_codigo_txt, '')) ILIKE '%abd%'
    OR LOWER(COALESCE(dsc_codigo_txt, '')) ILIKE '%pelve%'
    OR LOWER(COALESCE(cod_procedimento_txt, '')) ILIKE '%abd%'
    OR LOWER(COALESCE(cod_procedimento_txt, '')) ILIKE '%pelve%'
    )
    AND LOWER(COALESCE(exame_nr, '')) NOT LIKE '%angio%'
    AND LOWER(COALESCE(exame_nr, '')) NOT LIKE '%superior%'

    OR
        
      (
            lower(exame_nr) ILIKE '%colonoscopia%' OR 
            lower(exame_nr) ILIKE '%colono%' or
            LOWER(dsc_codigo_txt) ILIKE '%colonoscopia%' OR 
            LOWER(dsc_codigo_txt)ILIKE '%colono%'or
            LOWER(cod_procedimento_txt) ILIKE '%colonoscopia%' OR 
            LOWER(cod_procedimento_txt) ILIKE '%colono%'
        )

    )

    AND DataLaudoFinal IS NOT NULL
    AND DATE(DataLaudoFinal) BETWEEN date_sub(current_date(), 30)
                                AND date_sub(current_date(), 1)
),

dedup AS (
  SELECT
    f.*,
    ROW_NUMBER() OVER (
      PARTITION BY an
      ORDER BY TIMESTAMP(DataLaudoFinal) DESC
    ) AS rn,
    CASE WHEN ROW_NUMBER() OVER (
             PARTITION BY an
             ORDER BY TIMESTAMP(DataLaudoFinal) DESC
         ) = 1
         THEN 1 ELSE 0 END AS an_duplicado
  FROM filtrada f
)
SELECT
  idunidade,
  unidade,
  id_pct,
  paciente,
  telefonePacienteDDD,
  telefonePaciente,
  sexo,
  nascimento,
  idade_anos,
  cpf,
  modalidade,
  exame,
  num_pedido_integracao,
  nme_regional_hospital,
  nme_convenio,
  dataexame,
  tipoexame,
  an,
  an_duplicado,
  idstatus,
  -- 4 AS idstatus,                 -- mantido
  -- idLaudo,                       
  DataLaudoFinal AS DataLaudoLiberado,  -- >>> usa a data escolhida
  Laudo,
  -- current_timestamp() as datCarga
  CAST(from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo') AS TIMESTAMP_NTZ) AS dataExecucaoModelo
FROM dedup
""")

In [0]:
# spark.sql(f"""
# CREATE OR REPLACE TEMP VIEW {VW_BALIZADORA} AS
# WITH base AS (
#   SELECT
#       e.id_unidade                                     AS idunidade,
#       e.nme_hospital                                   AS unidade,
#       p.id_paciente                                    AS id_pct,
#       p.cli_nome                                       AS paciente,
#       p.cli_genero                                     AS sexo,
#       p.cli_data_nascimento                            AS nascimento,
#       p.doc_cpf                                        AS cpf,
#       p.cli_telefone_numero                            AS telefonePaciente,
#       p.cli_telefone_ddd                               AS telefonePacienteDDD,
#       e.tp_procedimento                                AS procedimento,
#       e.num_pedido_integracao                          AS num_pedido_integracao,
#       e.nme_regional_hospital                          AS nme_regional_hospital,
#       e.nme_convenio                                   AS nme_convenio,

#       regexp_replace(lower(coalesce(e.proced_descricao_ajustado, cast(e.cod_procedimento as string), '')), '[\\u00A0\\u2007\\u202F]', ' ') as exame_nr,

#       regexp_replace(lower(coalesce(e.dsc_codigo, '')), '[\\u00A0\\u2007\\u202F]', ' ') AS dsc_codigo_txt,
#       regexp_replace(lower(coalesce(cast(e.cod_procedimento as string), '')), '[\\u00A0\\u2007\\u202F]', ' ') AS cod_procedimento_txt,

#       /*calculo o numero de meses entre as duas datas e divido por 12 para ter o valor em anos*/
#       FLOOR(months_between(DATE(e.dt_dia_exame), DATE(p.cli_data_nascimento)) / 12) AS idade_anos,
#       CASE
#         WHEN exame_nr RLIKE '(tomografia|angio|tc|tomo)'
#           THEN 'CT'
#         WHEN exame_nr RLIKE '(colonoscopia|colono)' AND exame_nr NOT RLIKE '(tc|tomografia|ressonancia|rx)'
#           THEN 'COL'
#         ELSE 'IND'
#       END                                              AS modalidade,
#       translate(
#         regexp_replace(
#           lower(coalesce(exame_nr, cast(e.cod_procedimento as string), '')),
#           '[\\u00A0\\u2007\\u202F]', ' '
#         ),
#         'áàãâäéèêëíìîïóòõôöúùûüç',
#         'aaaaaeeeeiiiiooooouuuuc'
#       ) AS exame,

#       e.dt_dia_exame                                   AS dataexame,
#       translate(
#         regexp_replace(
#           lower(coalesce(e.nme_origem_exame, e.tp_proced_classe_encontro, '')),
#           '[\\u00A0\\u2007\\u202F]', ' '
#         ),
#         'áàãâäéèêëíìîïóòõôöúùûüç',
#         'aaaaaeeeeiiiiooooouuuuc'
#       ) AS tipoexame,

#       CAST(e.id_exame AS STRING)                       AS an,

#       e.dt_liberacao_laudo                             AS DataLaudoLiberado,
#       array_join(
#         transform(e.proced_lista_exames, x -> x.laudo_transformado),
#         '\n'
#       ) AS Laudo,
#       COALESCE(e.dt_liberacao_laudo, e.dt_previsao_laudo ) AS DataLaudoFinal,
#       CASE
#         WHEN e.dt_liberacao_laudo IS NOT NULL THEN 5
#         WHEN e.dt_previsao_laudo  IS NOT NULL THEN 4
#         ELSE 0
#       END AS idstatus

#   FROM gold_corporativo_ia.corporativo.tb_gold_mov_exame   e
#   LEFT JOIN gold_corporativo_ia.corporativo.tb_gold_mov_paciente p
#          ON p.id_paciente = e.id_patient
# ),

# filtrada AS (
#   SELECT *
#   FROM base
#   WHERE
#     UPPER(trim(procedimento)) IN  ('IMG', 'IMA')
#     AND(
#     (
#       LOWER(exame_nr) ILIKE '%tomografia%'
#       OR LOWER(exame_nr) ILIKE '%tc%'
#       OR LOWER(dsc_codigo_txt) ILIKE '%tomografia%'
#       OR LOWER(dsc_codigo_txt) ILIKE '%tc%'
#       OR LOWER(cod_procedimento_txt) ILIKE '%tomografia%'
#       OR LOWER(cod_procedimento_txt) ILIKE '%tc%'
#     )
#     AND(
#       LOWER(exame_nr) ILIKE '%abd%'
#       OR LOWER(exame_nr) ILIKE '%pelve%'
#       OR LOWER(dsc_codigo_txt) ILIKE '%abd%'
#       OR LOWER(dsc_codigo_txt) ILIKE '%pelve%'
#       OR LOWER(cod_procedimento_txt) ILIKE '%abd%'
#       OR LOWER(cod_procedimento_txt) ILIKE '%pelve%'
#     )
#     AND LOWER(exame_nr) not ilike '%angio%'
#     AND LOWER(exame_nr) not ilike '%superior%'
#     AND LOWER(dsc_codigo_txt) not ilike '%angio%'
#     AND LOWER(dsc_codigo_txt) not ilike '%superior%' 
#     AND LOWER(cod_procedimento_txt) not ilike '%angio%'
#     AND LOWER(cod_procedimento_txt) not ilike '%superior%'
    
#     OR
        
#       (
#             lower(exame_nr) ILIKE '%colonoscopia%' OR 
#             lower(exame_nr) ILIKE '%colono%' or
#             LOWER(dsc_codigo_txt) ILIKE '%colonoscopia%' OR 
#             LOWER(dsc_codigo_txt)ILIKE '%colono%'or
#             LOWER(cod_procedimento_txt) ILIKE '%colonoscopia%' OR 
#             LOWER(cod_procedimento_txt) ILIKE '%colono%'
#         )

#     )

#     AND DataLaudoFinal IS NOT NULL
#     AND DATE(DataLaudoFinal) BETWEEN date_sub(current_date(), 30)
#                                  AND date_sub(current_date(), 1) 
# ),

# dedup AS (
#   SELECT
#     f.*,
#     ROW_NUMBER() OVER (
#       PARTITION BY an
#       ORDER BY TIMESTAMP(DataLaudoFinal) DESC
#     ) AS rn,
#     CASE WHEN ROW_NUMBER() OVER (
#              PARTITION BY an
#              ORDER BY TIMESTAMP(DataLaudoFinal) DESC
#          ) = 1
#          THEN 1 ELSE 0 END AS an_duplicado
#   FROM filtrada f
# )
# SELECT
#   idunidade,
#   unidade,
#   id_pct,
#   paciente,
#   telefonePacienteDDD,
#   telefonePaciente,
#   sexo,
#   nascimento,
#   idade_anos,
#   cpf,
#   modalidade,
#   exame,
#   num_pedido_integracao,
#   nme_regional_hospital,
#   nme_convenio,
#   dataexame,
#   tipoexame,
#   an,
#   an_duplicado,
#   idstatus,
#   -- 4 AS idstatus,                 -- mantido
#   -- idLaudo,                       
#   DataLaudoFinal AS DataLaudoLiberado,  -- >>> usa a data escolhida
#   Laudo,
#   -- current_timestamp() as datCarga
#   CAST(from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo') AS TIMESTAMP_NTZ) AS dataExecucaoModelo
# FROM dedup
# """)

In [0]:
# %sql
# select
#   distinct(exame),
#   count(*) as qt_exame
# from dev_vw_dii_balizador
# group by exame
# order by qt_exame desc

In [0]:
spark.sql(f"""CREATE TABLE IF NOT EXISTS {SCHEMA}.{OUTPUT_TABLENAME}
            USING DELTA
            AS
            SELECT *,
            CAST(NULL AS STRING) AS laudo_tratado
            FROM {VW_BALIZADORA}
            WHERE 1 = 0
            """)

In [0]:
df= spark.sql(f"""
SELECT
    s.*
FROM {VW_BALIZADORA} s
LEFT ANTI JOIN ia.{OUTPUT_TABLENAME} t
    ON t.an       = s.an
""").toPandas()

In [0]:
df.display()

In [0]:
# spark.sql(f"""
# ALTER TABLE ia.{OUTPUT_TABLENAME}
# ADD COLUMNS (
#   num_pedido_integracao STRING,
#   nme_regional_hospital STRING,
#   nme_convenio STRING)
# """)

#to_plain

In [0]:
# ==== Stdlib ====
import re, unicodedata, html as _html, json
from typing import List, Any

# ==== Numérico / Dados ====
import pandas as pd
pd.set_option('display.float_format', lambda x: '%.5f' % x)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 2000)

# ==== RTF -> Texto (OK no PyPI) ====
from striprtf.striprtf import rtf_to_text

# ==== HTML -> Texto ====
from lxml.html import fromstring
import html2text

# ==== Unicode/Mojibake ====
import ftfy

from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

In [0]:
def norm(s: str) -> str:
    s = s.lower().strip()
    s = ''.join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != 'Mn')
    s = re.sub(r"\s+", " ", s)
    return s

In [0]:
#------- ALTERAÇÕES PARA HTML -------
_HTML_BRK_TAGS = re.compile(r'</?(p|div|br|li|tr|h[1-6])\b[^>]*>', re.IGNORECASE)
_HTML_TAGS     = re.compile(r'<[^>]+>')
_HTML_SCRIPT   = re.compile(r'<(script|style)\b[^>]*>.*?</\1\s*>', re.IGNORECASE|re.DOTALL)
_HTML_DATAIMG  = re.compile(r'data:image/[^;]+;base64,[A-Za-z0-9+/=\s]+', re.IGNORECASE)
_HTML_JUNK_ATTR= re.compile(r'\sdata-[\w\-]+="[^"]*"')
_HTML_JUNK_CLS = re.compile(r'\sclass="[^"]*"')
_HTML_WIDGET   = re.compile(r'\s(?:aria-[\w\-]+|role|tabindex|contenteditable|spellcheck|draggable|style|width|height)="[^"]*"', re.IGNORECASE)

def html_to_plain(s: str) -> str:
    if not s:
        return ""

    # normaliza EOL
    s = s.replace("\r\n", "\n").replace("\r", "\n")

    # Parse/limpeza estrutural com lxml (remove script/style/noscript)
    try:
        doc = fromstring(s)
        for bad in doc.xpath('//script|//style|//noscript'):
            bad.drop_tree()

        body = doc.find('body')
        node = body if body is not None else doc
        cleaned_html = lxml.html.tostring(node, encoding="unicode", method="html")
    except Exception:
        cleaned_html = s  # fallback: deixa o html2text tentar

    # HTML -> texto preservando parágrafos/quebras
    h = html2text.HTML2Text()
    h.body_width = 0                 # não “wrapa”
    h.single_line_break = True       # <br> vira \n
    h.ignore_links = True
    h.ignore_images = True
    h.ignore_emphasis = True
    h.unicode_snob = True

    try:
        s = h.handle(cleaned_html)
    except Exception:
        # fallback extremo: extrai texto puro do lxml
        try:
            s = fromstring(cleaned_html).text_content()
        except Exception:
            s = cleaned_html

    # decodifica entidades (&nbsp;, &ccedil;, etc.)
    s = _html.unescape(s)

    # limpa invisíveis e normaliza unicode
    s = s.replace("\u200b", "").replace("\u200c", "").replace("\xa0", " ")
    s = unicodedata.normalize("NFKC", s)

    # colagens comuns do editor
    s = re.sub(r'[ \t]{2,}', ' ', s)
    s = re.sub(r' ?/ ?', ' / ', s)
    s = re.sub(r'\n[ \t]+', '\n', s)

    # linhas em branco excessivas
    s = re.sub(r'\n{3,}', '\n\n', s)

    # strip final por linha e geral
    lines = [ln.rstrip() for ln in s.splitlines()]
    s = "\n".join(lines).strip()

    # correções pontuais vistas no corpus Tasy/CKE
    s = s.replace("Hospita\u200bl", "Hospital")
    s = re.sub(r'\bColonosocopia\b', 'Colonoscopia', s, flags=re.IGNORECASE)
    s = re.sub(r'\bredundancia\b', 'redundância', s, flags=re.IGNORECASE)

    return s


In [0]:
def _rtf_to_text_fallback(rtf: str, debug: bool = False) -> str:
    if not rtf:
        return ""

    import re, unicodedata

    # ---- helpers locais ----
    def _undo_py_escapes(s: str) -> str:
        # reverte escapes que o Python injeta quando a string não é raw
        return (s
            .replace("\x0c", r"\f")   # form feed -> \f
            .replace("\t",  r"\tab")  # tab -> \tab (token RTF real)
        )

    def _strip_rtf_metadata_top(s: str) -> str:
        # Remove SOMENTE blocos padrão logo no topo (primeiros ~8 KB)
        head = s[:8192]
        tail = s[8192:]

        # cada bloco: remove no máximo 1 ocorrência por padrão
        patterns = [
            r"{\\\*?\\fonttbl\b[^{}]*}",
            r"{\\\*?\\colortbl\b[^{}]*}",
            r"{\\\*?\\stylesheet\b[^{}]*}",
            r"{\\\*?\\listtable\b[^{}]*}",
            r"{\\\*?\\listoverridetable\b[^{}]*}",
            r"{\\\*?\\generator\b[^{}]*}",
            r"{\\\*?\\info\b[^{}]*}",
        ]
        for pat in patterns:
            head = re.sub(pat, " ", head, flags=re.IGNORECASE)

        return head + tail

    def _drop_rtf_meta_lines(text: str) -> str:
        out = []
        only_words_semicolon_rx = re.compile(r'^[A-Za-z0-9 /+\-\*\u00C0-\u017F]+;\s*$')
        font_tokens = r"(unicode|opensymbol|wingdings|monospaced|serif|sans|arial|calibri|times|courier)"
        # aceita " * " OU "\* " antes do nome da fonte
        font_line_rx = re.compile(
            rf'^\s*[A-Za-z0-9 \-\u00C0-\u017F]+(?:\s\\?\*\s*)?(?:{font_tokens})(?:\s*[;:]?)\s*$', re.IGNORECASE)
        tiny_meta_rx = re.compile(r'^\s*(default|\*jword2|\* generator|\* info)\s*[;:]?\s*$', re.IGNORECASE)

        for ln in text.splitlines():
            lns = ln.strip()
            if not lns:
                out.append(ln); continue
            if only_words_semicolon_rx.match(lns):
                continue
            if font_line_rx.match(lns):
                continue
            if tiny_meta_rx.match(lns):
                continue
            # fallback bem específico: linha curta com qualquer token de fonte + ';'
            if ';' in lns and re.search(font_tokens, lns, re.IGNORECASE) and len(lns) <= 80:
                continue
            out.append(ln)
        return "\n".join(out).strip()

    # -------------------------

    s = _undo_py_escapes(rtf)
    # normaliza EOL cedo
    s = s.replace("\r\n", "\n").replace("\r", "\n")

    # pega só se realmente parece RTF
    if not s.lstrip().startswith("{\\rtf"):
        # não é RTF: devolve como veio
        return s.strip()

    # tira metadados do topo de forma conservadora
    s = _strip_rtf_metadata_top(s)

    if debug:
        print("--- after strip_top ---")
        print(s[:1000])

    # 1) CP1252: \'hh
    def _hex_sub(m):
        try:
            return bytes([int(m.group(1), 16)]).decode('cp1252', errors='ignore')
        except Exception:
            return ""
    s = re.sub(r"\\'([0-9a-fA-F]{2})", _hex_sub, s)

    # 2) Unicode: \uNNNN?
    def _u_sub(m):
        try:
            code = int(m.group(1))
            if code < 0:
                code += 65536
            return chr(code)
        except Exception:
            return ""
    s = re.sub(r"\\u(-?\d+)\??", _u_sub, s)

    # 3) Quebras e tabs
    s = re.sub(r"\\par[d]?\b", "\n", s, flags=re.IGNORECASE)
    s = re.sub(r"\\line\b", "\n", s, flags=re.IGNORECASE)
    s = re.sub(r"\\tab\b", "\t", s, flags=re.IGNORECASE)

    # 4) Remove destinos conhecidos (restantes, pontuais)
    s = re.sub(r"{\\\*?\\(fonttbl|colortbl|stylesheet|info|generator|listtable|listoverridetable)[^}]*}",
               " ", s, flags=re.IGNORECASE)

    # 5) Remover comandos de controle (apenas os que começam com barra)
    #    Mais conservador: exige backslash + letras (com opcional número)
    #s = re.sub(r"\\[a-zA-Z]+-?\d*(?:\s|;)?", " ", s)
    s = re.sub(r"\\[a-zA-Z]+-?\d*\s?", " ", s)

    # 6) Escapes literais e chaves
    s = s.replace("\\{", "{").replace("\\}", "}").replace("\\\\", "\\")
    s = re.sub(r"[{}]", " ", s)  # neste ponto já decodificamos conteúdo útil

    # 7) Plano B: 'hh sem barra (alguns dumps)
    s = re.sub(r"(?<!\\)'([0-9a-fA-F]{2})", _hex_sub, s)

    # 8) Limpa tokens RTF residuais comuns (bem restritivo)
    s = re.sub(r"\b(?:rtf\d*|ansi|ansicpg\d+|deftab\d+|paper[wh]\d+|marg[tlrb]\d+|headery\d+|footery\d+|colsx?\d+|snext\d+|fs\d+|cf\d+|pard|plain|qc|ql|itap\d+|viewkind\d+|uc\d+|sa\d+|sl\d+|slmult\d+|lang\d+|kerning\d+|ulnone|b0|i0|f\d+|fs\d+|cf\d+|kerning\d+|deff\d+|colortbl|fonttbl|stylesheet|info|generator|listtable|listoverridetable)\b",
           " ", s, flags=re.IGNORECASE)

    # 9) Normalização
    s = unicodedata.normalize("NFKC", s)
    # remove invisíveis
    s = s.replace("\u200b", "").replace("\u200c", "").replace("\xa0", " ")
    # espaços
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\s*\n\s*", "\n", s)
    s = re.sub(r"\n{3,}", "\n\n", s).strip()

    # 10) Derruba linhas-meta finais
    s = _drop_rtf_meta_lines(s)

    # Garantia mínima: se sobrar vazio, devolve original “bruto” decodificado por CP1252
    if not s:
        s = bytes(rtf, "latin1", errors="ignore").decode("cp1252", errors="ignore").strip()

    return s

In [0]:
def remove_final_laudo(texto: str) -> str:
    if not texto:
        return ""

    # --- REMOÇÕES GLOBAIS INLINE (apareçam onde aparecerem) ---

    # 1) "Laudo Gerado/Lib[e]rado por Sistema Especialista."
    texto = re.sub(
        r'(?i)\blaudo\s+(?:gerado|liberado)\s+por\s+um?\s*sistema\s+especialista\b\.?',
        ' ',
        texto
    )

    # 2) "Para visualização do conteúdo do laudo, favor/… acessar/acesse … Viewer/opção imagem/imagem disponível"
    texto = re.sub(
        r'(?i)\bpara\s+visualiza\w*\s+(?:o\s+conte[uú]do\s+do\s+)?laudo\b.*?\b(?:favor\s+)?(?:acesse|acessar)\b.*?\b(?:viewer|op[cç][aã]o\s+imagem|imagem\s+dispon[ií]vel)\b\.?',
        ' ',
        texto
    )

    # 3) "Laudo pode não estar completo na visualização em RTF/Imagem"
    texto = re.sub(
        r'(?i)\blaudo\s+pode\s+na[oã]\s+estar\s+completo\s+na\s+visualiza\w+\s+em\s+(?:rtf|imagem)\b\.?',
        ' ',
        texto
    )

    # 4) "Referências: - Endoscopic Classification Review Group..." (e variações)
    # Busca por "referencia(s)" e remove o restante do texto.
    texto = re.sub(
        r'(?i)\brefer[eê]ncia[s]?\s*[:\-\.]?.*$',
        ' ',
        texto,
        flags=re.DOTALL
    )

    # --- REMOÇÃO POR SENTENÇA (linhas inteiras de aviso/rodapé) ---
    pats_norm = [
        r"laudo\s+pode\s+nao\s+estar\s+completo.*?\b(rtf|imagem)\b",
        r"para\s+visualiza\w+.*?\b(rtf|imagem|sistema|webris|viewer)\b",
        r"acessar\s+a\s+op[cç][aã]o\s+imagem",
        r"sistema\s+especialista",
        r"sistema\s+da\s+radiologia\s+webris",
        # novas combinações explícitas:
        r"este\s+laudo\s+foi\s+liberado\s+por\s+um?\s+sistema\s+especialista",
        r"favor\s+acessar\s+a\s+op[cç][aã]o\s+imagem\s+dispon[ií]vel",
        r"visualiza\w+\s+do\s+conteudo\s+do\s+laudo",
        r"laudo\s+pode\s+nao\s+estar\s+completo\s+na\s+visualiza\w+\s+em\s+rtf",
    ]

    rx = re.compile(
        r"^(?:\s*(?:{})(?:[.!?])?\s*)$".format("|".join(pats_norm)),
        re.IGNORECASE
    )

    # --- ALTERAÇÃO: preservar quebras de linha ---
    lines_in = texto.splitlines()
    lines_out = []

    for line in lines_in:
        if not line.strip():
            lines_out.append(line)
            continue

        # separa sentenças só dentro da linha (não quebra em \n)
        sents = re.split(r'(?<=[.!?])\s+', line.strip())
        kept = []
        for s in sents:
            s_norm = norm(s)
            if not rx.match(s_norm):
                kept.append(s)

        lines_out.append(" ".join(kept).strip())

    out = "\n".join(lines_out)

    # mantém os pós-processamentos originais
    out = re.sub(r'\s{2,}', ' ', out)
    out = re.sub(r'\s+([.!?,;:])', r'\1', out)
    return out.strip()

# --- Pós-processamento PT-BR para laudos (leve e seguro) ---
_UPPER_SPACED_RX = re.compile(r'^(?:[A-ZÁ-Ú]\s+){2,}[A-ZÁ-Ú]$', re.UNICODE)

In [0]:
def to_plain(s: str) -> str:
    """
    Conversão robusta (RTF/HTML/texto) -> texto limpo:
      1) detecta RTF/HTML e usa parser dedicado (striprtf / lxml / html2text)
      2) ftfy para arrumar unicode/mojibake
      3) normalizações e correções de espaços intra-palavra/acento
      4) limpeza de boilerplate típico de RTF/HTML
      5) remoção de rodapés conhecidos e ajuste de pontuação
    """
    if not s:
        return ""

    raw = s.replace("\r\n", "\n").replace("\r", "\n")

    # --- NOVO: alguns dumps chegam com escapes literais "\r\n" e "\n" (texto, não EOL real) ---
    if "\\r\\n" in raw:
        raw = raw.replace("\\r\\n", "\n")

    # troca "\n" literal só quando é claramente quebra (antes de "{", "\" , espaço ou fim)
    if "\\n" in raw:
        raw = re.sub(r'\\n(?=[\s{\\}]|$)', "\n", raw)

    # --- 1) Detectar formato ---
    head = raw[:512].lstrip()
    is_rtf  = head.startswith("{\\rtf") or "\\rtf" in head.lower()[:128]
    is_html = bool(re.search(r"</?(html|head|body|div|p|span|br|table|tr|td|img)\b", raw[:2000], re.I))

    txt = raw
    try:
        if is_rtf:
            try:
                # tenta pandoc se existir (melhor preservação de estrutura em alguns RTF)
                try:
                    import pypandoc
                    txt = pypandoc.convert_text(raw, 'plain', format='rtf', extra_args=['--wrap=none'])
                except Exception:
                    txt = rtf_to_text(raw)
            except Exception:
                txt = _rtf_to_text_fallback(raw)
        elif is_html:
            txt = html_to_plain(raw)
        else:
            txt = raw
    except Exception:
        # fallback bruto caso algo quebre
        txt = raw

    # --- 2) Unicode/encodes (corrige mojibake, NBSP etc.) ---
    try:
        txt = ftfy.fix_text(txt)
    except Exception:
        pass

    # invisíveis e normalização
    txt = (txt.replace("\u200b", "")
              .replace("\u200c", "")
              .replace("\xa0", " "))
    txt = unicodedata.normalize("NFKC", txt)

    # --- 3) Colapsar cabeçalhos ALL CAPS espaçados (CON C LU S Ã O) ---
    RX_CAPS_LINE   = re.compile(r'^(?:[A-ZÁ-Ú]\s+){2,}[A-ZÁ-Ú]$', re.UNICODE)
    RX_CAPS_INLINE = re.compile(r'(?<!\w)([A-ZÁ-Ú](?:\s+[A-ZÁ-Ú]){2,})(?!\w)', re.UNICODE)
    def _collapse_caps(line: str) -> str:
        if RX_CAPS_LINE.match(line.strip()):
            return line.replace(" ", "")
        return RX_CAPS_INLINE.sub(lambda m: m.group(1).replace(" ", ""), line)
    txt = "\n".join(_collapse_caps(ln) for ln in txt.splitlines())

    # --- 4) Remover prefixo numérico de OCR grudado (ex.: "6reto" -> "reto"), sem afetar dosagens ---
    UNITS = {
        "mg","g","kg","mcg","ug","μg","ml","l","dl","cl","mm","cm","m","km",
        "bpm","mmhg","pa","ua","u/l","ui","iu","u","meq","mol","mmol","nmol",
        "na","k","cl","co2","o2","sat","spo2","fio2","cr","vdrl","ph"
    }
    RX_DOSAGE = re.compile(r'(?i)^\d{1,3}\s*(?:' + r'|'.join(sorted(UNITS, key=len, reverse=True)) + r')\b')
    RX_FLOW   = re.compile(r'(?i)^\d{1,3}\s*[lL]\s*/\s*min\b')
    RX_OCR_PREFIX = re.compile(r'(?i)\b(\d{1,2})([A-Za-zÁ-Úá-ú]{3,})\b')
    def _strip_digit_prefix(m: re.Match) -> str:
        tok = m.group(0)
        if RX_DOSAGE.match(tok) or RX_FLOW.match(tok) or m.group(2).lower() in UNITS:
            return tok
        return m.group(2)
    txt = RX_OCR_PREFIX.sub(_strip_digit_prefix, txt)

    # --- 5) Corrigir espaços intra-palavra (acentos/ç quebrados) ---
    ACC = "ÁÉÍÓÚÂÊÎÔÛÃÕÄËÏÖÜáéíóúâêîôûãõäëïöüÇç"
    for _ in range(4):
        before = txt

        # letras acentuadas separadas do resto
        txt = re.sub(rf'(?i)\b([{ACC}])\s+([A-Za-z]{{1,30}})\b', r'\1\2', txt)
        txt = re.sub(rf'(?i)\b([A-Za-z]{{1,30}})\s+([{ACC}][A-Za-z]{{1,5}})\b', r'\1\2', txt)

        # trinca com acento no meio: "p ó s" -> "pós"
        txt = re.sub(rf'(?i)\b([A-Za-z])\s+([{ACC}])\s+([A-Za-z])\b', r'\1\2\3', txt)

        # sufixos comuns
        txt = re.sub(r'(?i)çã\s+o\b',  'ção',  txt)
        txt = re.sub(r'(?i)çõ\s+es\b', 'ções', txt)
        txt = re.sub(r'(?i)ã\s+o\b',   'ão',   txt)
        txt = re.sub(r'(?i)õ\s+es\b',  'ões',  txt)

        # hífen com espaços indevidos
        txt = re.sub(rf'(?i)\b([A-Za-z{ACC}]{{1,40}})\s*-\s*([A-Za-z{ACC}]{{1,40}})\b', r'\1-\2', txt)

        # casos de espaço antes de letras acentuadas dentro de palavra (cola sempre)
        txt = re.sub(rf'\b([A-Za-z]{{1,4}})\s+(?=[{ACC}])', r'\1', txt)
        txt = re.sub(rf'\b([{ACC}])\s+(?=[A-Za-z]{{1,3}}\b)', r'\1', txt)

        if txt == before:
            break

    # --- 6) Limpeza de boilerplate herdado de RTF/HTML ---
    lines = []
    for ln in txt.splitlines():
        s = ln.strip()
        sl = s.lower()
        if not s:
            lines.append(ln); continue
        # lixos clássicos de RTF/HTML
        if any(tok in sl for tok in ("fonttbl","colortbl","stylesheet","generator",
                                     "heading","listoverridetable","listtable")):
            continue
        if s.count(';') >= 2 and re.search(r'(arial|times|calibri|courier|symbol|wingdings)', sl):
            continue
        # linhas só de pontuação/traços
        if re.fullmatch(r'[;,\.\-–—\s]+', s):
            continue
        lines.append(ln)
    txt = "\n".join(lines)

    # --- 7) Espaços/pontuação finais e quebras ---
    txt = re.sub(r'[ \t]+', ' ', txt)
    txt = re.sub(r'\s*\n\s*', '\n', txt)
    txt = re.sub(r'\n{3,}', '\n\n', txt)
    txt = re.sub(r'\s+([.,;:])', r'\1', txt)
    txt = re.sub(r'([\(\[\{])\s+', r'\1', txt)
    txt = re.sub(r'\s+([\)\]\}])', r'\1', txt)

    # --- 8) Remover rodapés/avisos de visualização incompleta e limpar pontas ---
    txt = remove_final_laudo(txt).strip()
    return txt

In [0]:
df["laudo_tratado"] = df["Laudo"].map(to_plain)

%md
# Pós tratamento do laudo

In [0]:
from pyspark.sql.types import *

df["dataexame"] = pd.to_datetime(df["dataexame"], errors="coerce")
df["nascimento"] = pd.to_datetime(df["nascimento"], errors="coerce")
df["DataLaudoLiberado"] = pd.to_datetime(df["DataLaudoLiberado"], errors="coerce")
df["dataExecucaoModelo"] = pd.to_datetime(df["dataExecucaoModelo"], errors="coerce")

schema = StructType([
    StructField("idunidade", StringType(), True),
    StructField("unidade", StringType(), True),
    StructField("id_pct", StringType(), True),
    StructField("paciente", StringType(), True),
    StructField("telefonePacienteDDD", StringType(), True),
    StructField("telefonePaciente", StringType(), True),
    StructField("sexo", StringType(), True),
    StructField("nascimento", TimestampType(), True),
    StructField("idade_anos", LongType(), True),
    StructField("cpf", StringType(), True),
    StructField("modalidade", StringType(), True),
    StructField("exame", StringType(), True),
    StructField("num_pedido_integracao", StringType(), True),
    StructField("nme_regional_hospital", StringType(), True),
    StructField("nme_convenio", StringType(), True),
    StructField("dataexame", TimestampType(), True),
    StructField("tipoexame", StringType(), True),
    StructField("an", StringType(), True),
    StructField("an_duplicado", IntegerType(), True),
    StructField("idstatus", IntegerType(), True),
    StructField("DataLaudoLiberado", TimestampType(), True),
    StructField("Laudo", StringType(), True),
    StructField("dataExecucaoModelo", TimestampType(), True),
    StructField("laudo_tratado", StringType(), True)
])

df_spark = spark.createDataFrame(df, schema=schema)


In [0]:
df_spark.createOrReplaceTempView("vw")

In [0]:
df_spark.createOrReplaceTempView("vw")

In [0]:
%sql
CREATE or REPLACE temp VIEW wrk AS
select 
*
FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY an ORDER BY an) AS rn
    FROM vw
) t
where true
AND NOT (lower(laudo_tratado) RLIKE '\\{\\\\rtf') 
AND NOT (lower(laudo_tratado) RLIKE '<[^>]+>') 
AND lower(laudo_tratado) NOT LIKE '%laudo em pdf%' 
AND lower(laudo_tratado) NOT LIKE '%sistema especialista%'
and lower(laudo_tratado) NOT LIKE '%visualização em rtf%'
and laudo is not null
and rn = 1 


####Incrementação de dados diarios

In [0]:
spark.sql(f"""
INSERT INTO {SCHEMA}.{OUTPUT_TABLENAME} (
    idunidade,
    unidade,
    id_pct,
    paciente,
    telefonePacienteDDD,
    telefonePaciente,
    sexo,
    nascimento,
    idade_anos,
    cpf,
    modalidade,
    exame,
    num_pedido_integracao,
    nme_regional_hospital,
    nme_convenio,
    dataexame,
    tipoexame,
    an,
    an_duplicado,
    idstatus,
    DataLaudoLiberado,
    Laudo,
    dataExecucaoModelo,
    laudo_tratado
)
SELECT
    idunidade,
    unidade,
    id_pct,
    paciente,
    telefonePacienteDDD,
    telefonePaciente,
    sexo,
    nascimento,
    idade_anos,
    cpf,
    modalidade,
    exame,
    num_pedido_integracao,
    nme_regional_hospital,
    nme_convenio,
    dataexame,              
    tipoexame,
    an,
    an_duplicado,
    idstatus,
    DataLaudoLiberado,
    Laudo,
    dataExecucaoModelo,
    laudo_tratado
FROM wrk
""")

####Consultas

In [0]:
%sql
select 
count(an) as total,
cast(date_trunc('day', dataexame) as date) as data_exame
from wrk
where true
AND NOT (lower(laudo_tratado) RLIKE '\\{\\\\rtf') 
AND NOT (lower(laudo_tratado) RLIKE '<[^>]+>') 
AND lower(laudo_tratado) NOT LIKE '%laudo em pdf%' 
AND lower(laudo_tratado) NOT LIKE '%sistema especialista%'
and modalidade = "COL"
group by data_exame
order by data_exame

In [0]:
dbutils.notebook.exit("Final Notebook")